<a href="https://colab.research.google.com/github/maruwagner893-droid/cloud-provider-analytics/blob/main/notebooks/02_pipeline_etapa2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("cloud-provider-analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("Versión de Spark:", spark.version)

Versión de Spark: 4.0.4


In [3]:
import os

REPO = "/content/cloud-provider-analytics"

if not os.path.isdir(REPO):
    !git clone https://github.com/maruwagner893-droid/cloud-provider-analytics {REPO}

!unzip -q -o {REPO}/data/cloud_provider_challenge_dataset_v1.zip -d {REPO}/data/extracted

Cloning into '/content/cloud-provider-analytics'...
remote: Enumerating objects: 818, done.
remote: Counting objects: 100% (633/633), done.
remote: Compressing objects: 100% (485/485), done.
remote: Total 818 (delta 365), reused 148 (delta 120), pack-reused 185 (from 1)
Receiving objects: 100% (818/818), 3.85 MiB | 8.80 MiB/s, done.
Resolving deltas: 100% (447/447), done.


In [4]:
LANDING = os.path.join(REPO, "data/extracted/datalake/landing")
print("Landing:", LANDING)
print(sorted(os.listdir(LANDING)))

Landing: /content/cloud-provider-analytics/data/extracted/datalake/landing
['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'usage_events_stream', 'users.csv']


In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, DateType, DoubleType
)

schema_customers = StructType([
    StructField("org_id",           StringType(),  True),
    StructField("org_name",         StringType(),  True),
    StructField("industry",         StringType(),  True),
    StructField("hq_region",        StringType(),  True),
    StructField("plan_tier",        StringType(),  True),
    StructField("is_enterprise",    BooleanType(), True),
    StructField("signup_date",      DateType(),    True),
    StructField("sales_rep",        StringType(),  True),
    StructField("lifecycle_stage",  StringType(),  True),
    StructField("marketing_source", StringType(),  True),
    StructField("nps_score",        DoubleType(),  True),
])

customers = (
    spark.read
    .option("header", True)
    .schema(schema_customers)
    .csv(os.path.join(LANDING, "customers_orgs.csv"))
)

customers.printSchema()
customers.show(5, truncate=False)

# Verificación: tiene que coincidir con tu notebook
print("Filas:", customers.count())                                                 # 80
print("signup_date nulos:", customers.filter(F.col("signup_date").isNull()).count())  # 0
print("nps_score nulos:", customers.filter(F.col("nps_score").isNull()).count())      # 11

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)

+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  |Education    |sa-east   |standard |true         |2025-05-26 |rep_

In [6]:
BRONZE = os.path.join(REPO, "data/bronze")

customers_bronze = (
    customers
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
)

(
    customers_bronze
    .coalesce(1)                      # 80 filas: un solo archivo, sin archivos chicos
    .write.mode("overwrite")          # re-ejecutable: pisa el resultado anterior
    .parquet(os.path.join(BRONZE, "customers_orgs"))
)

# Verificación: leer lo que se guardó
check = spark.read.parquet(os.path.join(BRONZE, "customers_orgs"))
check.printSchema()
check.select("org_id", "ingest_ts", "source_file").show(3, truncate=False)
print("Filas en Bronze:", check.count())   # 80

!ls -la {BRONZE}/customers_orgs

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

+------------+--------------------------+------------------+
|org_id      |ingest_ts                 |source_file       |
+------------+--------------------------+------------------+
|org_xaji0y6d|2026-10-22 23:29:07.049613|customers_orgs.csv|
|org_pbhsahxt|2026-10-22 23:29:07.049613|customers_orgs.csv|
|org_hv3a3zmf|2026-10-22 23:29:07.049613|customers_orgs.csv|
+------------+--------------------------+------------------+
only showing top

In [7]:
schema_users = StructType([
    StructField("user_id",    StringType(),  True),
    StructField("org_id",     StringType(),  True),
    StructField("email",      StringType(),  True),
    StructField("role",       StringType(),  True),
    StructField("active",     BooleanType(), True),
    StructField("created_at", DateType(),    True),
    StructField("last_login", DateType(),    True),
])

schema_billing = StructType([
    StructField("invoice_id",           StringType(), True),
    StructField("org_id",               StringType(), True),
    StructField("month",                DateType(),   True),
    StructField("subtotal",             DoubleType(), True),
    StructField("credits",              DoubleType(), True),
    StructField("taxes",                DoubleType(), True),
    StructField("currency",             StringType(), True),
    StructField("exchange_rate_to_usd", DoubleType(), True),
])

def ingest_master(name, schema, key):
    """Landing (CSV) -> Bronze (Parquet): tipifica, deduplica por clave y agrega metadatos."""
    df = (
        spark.read.option("header", True).schema(schema)
        .csv(os.path.join(LANDING, f"{name}.csv"))
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_name"))
    )
    n_in = df.count()
    df = df.dropDuplicates([key])
    n_out = df.count()
    out = os.path.join(BRONZE, name)
    df.coalesce(1).write.mode("overwrite").parquet(out)
    print(f"{name}: leídas={n_in}  duplicadas quitadas={n_in - n_out}  guardadas={n_out}  -> {out}")

ingest_master("customers_orgs",  schema_customers, "org_id")
ingest_master("users",           schema_users,     "user_id")
ingest_master("billing_monthly", schema_billing,   "invoice_id")

# Verificación contra tu notebook
users_b   = spark.read.parquet(os.path.join(BRONZE, "users"))
billing_b = spark.read.parquet(os.path.join(BRONZE, "billing_monthly"))
print("users last_login nulos:", users_b.filter(F.col("last_login").isNull()).count())      # 139
print("billing credits nulos:", billing_b.filter(F.col("credits").isNull()).count())        # 137
print("billing subtotal < 0:", billing_b.filter(F.col("subtotal") < 0).count())             # 13

customers_orgs: leídas=80  duplicadas quitadas=0  guardadas=80  -> /content/cloud-provider-analytics/data/bronze/customers_orgs
users: leídas=800  duplicadas quitadas=0  guardadas=800  -> /content/cloud-provider-analytics/data/bronze/users
billing_monthly: leídas=240  duplicadas quitadas=0  guardadas=240  -> /content/cloud-provider-analytics/data/bronze/billing_monthly
users last_login nulos: 139
billing credits nulos: 137
billing subtotal < 0: 13


In [8]:
from pyspark.sql.types import TimestampType, IntegerType, LongType
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

schema_events = StructType([
    StructField("event_id",           StringType(),    True),
    StructField("timestamp",          TimestampType(), True),
    StructField("org_id",             StringType(),    True),
    StructField("resource_id",        StringType(),    True),
    StructField("service",            StringType(),    True),
    StructField("region",             StringType(),    True),
    StructField("metric",             StringType(),    True),
    StructField("value",              StringType(),    True),   # llega como número o como texto
    StructField("unit",               StringType(),    True),
    StructField("cost_usd_increment", DoubleType(),    True),
    StructField("schema_version",     IntegerType(),   True),
    StructField("carbon_kg",          DoubleType(),    True),   # solo en v2
    StructField("genai_tokens",       LongType(),      True),   # solo servicio genai
])

EVENTS_LANDING = os.path.join(LANDING, "usage_events_stream")
REPLAY = os.path.join(REPO, "data/replay/usage_events_stream")

events_all = spark.read.schema(schema_events).json(EVENTS_LANDING)

(
    events_all
    .repartitionByRange(120, "timestamp")      # 120 archivos, cada uno con un tramo de tiempo
    .sortWithinPartitions("timestamp")
    .write.mode("overwrite")
    .json(REPLAY)
)

# Verificación: los archivos tienen que quedar ordenados y sin solaparse
replay = (
    spark.read.schema(schema_events).json(REPLAY)
    .withColumn("f", F.col("_metadata.file_name"))
)
rangos = (
    replay.groupBy("f")
    .agg(F.count("*").alias("n"), F.min("timestamp").alias("desde"), F.max("timestamp").alias("hasta"))
    .orderBy("f")
)
rangos.show(5, truncate=False)

w = Window.orderBy("f")
solapes = (
    rangos.withColumn("desde_sig", F.lead("desde").over(w))
    .filter(F.col("hasta") > F.col("desde_sig"))
    .count()
)
print("Archivos:", rangos.count())
print("Eventos totales:", replay.count())                 # 43200
print("Archivos que se solapan con el siguiente:", solapes)   # 0

+---------------------------------------------------------+---+-------------------+-------------------+
|f                                                        |n  |desde              |hasta              |
+---------------------------------------------------------+---+-------------------+-------------------+
|part-00000-5c8173c6-5fec-438e-83e5-8ce7e32cebef-c000.json|381|2025-07-03 00:02:00|2025-07-03 13:19:00|
|part-00001-5c8173c6-5fec-438e-83e5-8ce7e32cebef-c000.json|368|2025-07-03 13:22:00|2025-07-04 01:11:00|
|part-00002-5c8173c6-5fec-438e-83e5-8ce7e32cebef-c000.json|371|2025-07-04 01:13:00|2025-07-04 13:38:00|
|part-00003-5c8173c6-5fec-438e-83e5-8ce7e32cebef-c000.json|359|2025-07-04 13:40:00|2025-07-05 01:24:00|
|part-00004-5c8173c6-5fec-438e-83e5-8ce7e32cebef-c000.json|351|2025-07-05 01:25:00|2025-07-05 12:22:00|
+---------------------------------------------------------+---+-------------------+-------------------+
only showing top 5 rows
Archivos: 120
Eventos totales: 43200
Arc

In [9]:
BRONZE_EVENTS = os.path.join(BRONZE, "usage_events_stream")
CHECKPOINT    = os.path.join(REPO, "data/checkpoints/bronze_events")

stream = (
    spark.readStream
    .schema(schema_events)
    .option("maxFilesPerTrigger", 1)          # un archivo por micro-lote
    .json(REPLAY)
)

bronze_stream = (
    stream
    .withColumn("value", F.col("value").cast("double"))        # texto numérico -> número
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("year", F.year("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withWatermark("timestamp", "10 minutes")                   # decisión de tu documento
    .dropDuplicatesWithinWatermark(["event_id"])                # deduplica por event_id
)

def run_bronze_events():
    q = (
        bronze_stream.writeStream
        .format("parquet")
        .outputMode("append")
        .option("path", BRONZE_EVENTS)
        .option("checkpointLocation", CHECKPOINT)
        .partitionBy("year", "month")                           # particiones de tu documento
        .trigger(availableNow=True)                             # procesa todo lo pendiente y termina
        .start()
    )
    q.awaitTermination()

def contar_bronze():
    d = spark.read.parquet(BRONZE_EVENTS)
    print("Eventos en Bronze:", d.count())
    d.groupBy("year", "month").count().orderBy("year", "month").show()

# Primera ejecución
run_bronze_events()
contar_bronze()

Eventos en Bronze: 43200
+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2025|    7|20880|
|2025|    8|22320|
+----+-----+-----+



In [10]:
import shutil, glob

def total_bronze():
    return spark.read.parquet(BRONZE_EVENTS).count()

# A) Re-ejecutar sin archivos nuevos: el checkpoint evita reprocesar
antes = total_bronze()
run_bronze_events()
despues = total_bronze()
print(f"A) Re-ejecución sin archivos nuevos -> antes={antes}  después={despues}")

# B) Llega un archivo repetido (mismos event_id): no debe sumar eventos
ultimo = sorted(glob.glob(os.path.join(REPLAY, "part-*.json")))[-1]
repetido = os.path.join(REPLAY, "part-99999-repetido.json")
shutil.copy(ultimo, repetido)

antes = total_bronze()
run_bronze_events()
despues = total_bronze()
print(f"B) Llega un archivo repetido      -> antes={antes}  después={despues}")

os.remove(repetido)   # dejamos el replay como estaba

print("Checkpoint:", os.listdir(CHECKPOINT))

A) Re-ejecución sin archivos nuevos -> antes=43200  después=43200
B) Llega un archivo repetido      -> antes=43200  después=43200
Checkpoint: ['commits', 'metadata', 'state', '.metadata.crc', 'offsets', 'sources']


In [11]:
SILVER_EVENTS     = os.path.join(REPO, "data/silver/usage_events")
QUARANTINE_EVENTS = os.path.join(REPO, "data/quarantine/usage_events")
CHECKPOINT_SILVER = os.path.join(REPO, "data/checkpoints/silver_events")

customers_dim = (
    spark.read.parquet(os.path.join(BRONZE, "customers_orgs"))
    .select("org_id", "industry", F.col("hq_region").alias("org_hq_region"),
            "plan_tier", "is_enterprise")
)

unidad_por_metrica = (
    F.when(F.col("metric") == "cpu_hours", "hours")
     .when(F.col("metric") == "requests", "count")
     .when(F.col("metric") == "storage_gb_hours", "gb_hours")
)

def procesar_lote(batch_df, batch_id):
    motivo = (
        F.when(F.col("event_id").isNull(), "event_id_nulo")
         .when(F.col("cost_usd_increment") < -0.01, "cost_usd_increment_menor_a_-0.01")
    )
    marcado = batch_df.withColumn("quarantine_reason", motivo)

    invalidos = (
        marcado.filter(F.col("quarantine_reason").isNotNull())
        .withColumn("quarantine_ts", F.current_timestamp())
    )
    validos = marcado.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")

    silver = (
        validos
        .withColumn("unit_imputed", F.col("unit").isNull())
        .withColumn("unit", F.coalesce(F.col("unit"), unidad_por_metrica))
        .withColumn("value_missing", F.col("value").isNull())
        .withColumn("usage_date", F.to_date("timestamp"))
        .withColumn("requests",         F.when(F.col("metric") == "requests",         F.col("value")))
        .withColumn("cpu_hours",        F.when(F.col("metric") == "cpu_hours",        F.col("value")))
        .withColumn("storage_gb_hours", F.when(F.col("metric") == "storage_gb_hours", F.col("value")))
        .join(F.broadcast(customers_dim), on="org_id", how="left")
    )

    silver.write.mode("append").partitionBy("year", "month").parquet(SILVER_EVENTS)
    invalidos.write.mode("append").parquet(QUARANTINE_EVENTS)

bronze_schema = spark.read.parquet(BRONZE_EVENTS).schema
silver_stream = (
    spark.readStream.schema(bronze_schema)
    .option("maxFilesPerTrigger", 10)
    .parquet(BRONZE_EVENTS)
)

def run_silver_events():
    q = (
        silver_stream.writeStream
        .foreachBatch(procesar_lote)
        .option("checkpointLocation", CHECKPOINT_SILVER)
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()

run_silver_events()

# Verificación
s = spark.read.parquet(SILVER_EVENTS)
q = spark.read.parquet(QUARANTINE_EVENTS)
n_s, n_q = s.count(), q.count()
print("Silver:", n_s)                                              # 42989
print("Quarantine:", n_q)                                          # 211
print("% inválidos:", round(n_q / (n_s + n_q) * 100, 2))           # 0.49
s.groupBy("year", "month").count().orderBy("year", "month").show()  # 20778 / 22211
print("unit nulos en Silver:", s.filter(F.col("unit").isNull()).count())          # 0
print("unit imputada:", s.filter("unit_imputed").count())                         # 2069
print("value nulo:", s.filter("value_missing").count())                           # 873
q.groupBy("quarantine_reason").count().show(truncate=False)
q.select("event_id", "cost_usd_increment", "quarantine_reason", "source_file") \
 .orderBy("cost_usd_increment").show(5, truncate=False)                           # mínimo -154.46
s.select("event_id", "service", "metric", "unit", "requests", "cpu_hours",
         "storage_gb_hours", "industry", "plan_tier").show(5, truncate=False)

Silver: 42989
Quarantine: 211
% inválidos: 0.49
+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2025|    7|20778|
|2025|    8|22211|
+----+-----+-----+

unit nulos en Silver: 0
unit imputada: 2069
value nulo: 873
+--------------------------------+-----+
|quarantine_reason               |count|
+--------------------------------+-----+
|cost_usd_increment_menor_a_-0.01|211  |
+--------------------------------+-----+

+----------------+------------------+--------------------------------+---------------------------------------------------------+
|event_id        |cost_usd_increment|quarantine_reason               |source_file                                              |
+----------------+------------------+--------------------------------+---------------------------------------------------------+
|evt_t5brta67j5l3|-154.4608         |cost_usd_increment_menor_a_-0.01|part-00105-5c8173c6-5fec-438e-83e5-8ce7e32cebef-c000.json|
|evt_9ur456xoexl5|-19.8245          |cost_usd_increment

In [12]:
GOLD_USAGE = os.path.join(REPO, "data/gold/org_daily_usage_by_service")

silver = spark.read.parquet(SILVER_EVENTS)

org_daily_usage_by_service = (
    silver
    .groupBy("org_id", "usage_date", "service")
    .agg(
        F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
        F.sum("requests").alias("requests"),
        F.sum("cpu_hours").alias("cpu_hours"),
        F.sum("storage_gb_hours").alias("storage_gb_hours"),
        F.sum("genai_tokens").alias("genai_tokens"),
        F.sum("carbon_kg").alias("carbon_kg"),
        F.count("*").alias("n_events"),
    )
)

# Evidencia del shuffle (captura para evidence/)
org_daily_usage_by_service.explain(mode="formatted")

(
    org_daily_usage_by_service
    .coalesce(1)
    .write.mode("overwrite")
    .parquet(GOLD_USAGE)
)

# Verificación
gold = spark.read.parquet(GOLD_USAGE)
print("Filas Gold:", gold.count())                                        # 11050
claves = gold.select("org_id", "usage_date", "service").distinct().count()
print("Claves únicas (grano):", claves)                                   # 11050
print("Suma n_events:", gold.agg(F.sum("n_events")).first()[0])           # 42989
print("Costo total:", round(gold.agg(F.sum("daily_cost_usd")).first()[0], 2))   # ~148351.58
print("Requests total:", gold.agg(F.sum("requests")).first()[0])          # 2281020.0
print("genai_tokens total:", gold.agg(F.sum("genai_tokens")).first()[0])  # 2545481
print("Filas con genai_tokens:", gold.filter(F.col("genai_tokens").isNotNull()).count())  # 848
gold.orderBy("org_id", "usage_date", "service").show(5, truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [11]: [org_id#12822, service#12826, cost_usd_increment#12831, carbon_kg#12833, genai_tokens#12834L, usage_date#12839, requests#12840, cpu_hours#12841, storage_gb_hours#12842, year#12847, month#12848]
Batched: true
Location: InMemoryFileIndex [file:/content/cloud-provider-analytics/data/silver/usage_events]
ReadSchema: struct<org_id:string,service:string,cost_usd_increment:double,carbon_kg:double,genai_tokens:bigint,usage_date:date,requests:double,cpu_hours:double,storage_gb_hours:double>

(2) Project
Output [9]: [org_id#12822, service#12826, cost_usd_increment#12831, carbon_kg#12833, genai_tokens#12834L, usage_date#12839, requests#12840, cpu_hours#12841, storage_gb_hours#12842]
Input [11]: [org_id#12822, service#12826, cost_usd_increment#12831, carbon_kg#12833, genai_tokens#12834L, usage_date#128

In [13]:
TEST_LATE = os.path.join(REPO, "data/test_late")
shutil.rmtree(TEST_LATE, ignore_errors=True)       # empezar limpio

late_stream = (
    spark.readStream
    .schema(schema_events)
    .option("maxFilesPerTrigger", 1)               # un archivo por micro-lote
    .json(EVENTS_LANDING)                          # archivos ORIGINALES, sin ordenar
    .withWatermark("timestamp", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
)

q = (
    late_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", os.path.join(TEST_LATE, "out"))
    .option("checkpointLocation", os.path.join(TEST_LATE, "checkpoint"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

total = 43200
pasaron = spark.read.parquet(os.path.join(TEST_LATE, "out")).count()
descartados = total - pasaron
print("Eventos de entrada:", total)
print("Pasaron el watermark:", pasaron)                                  # ~371
print("Descartados por llegar tarde:", descartados)                      # ~42829
print("% descartado:", round(descartados / total * 100, 2))              # ~99.1

# Contraste: el replay ordenado (Bronze) no pierde nada
print("Bronze con replay ordenado:", spark.read.parquet(BRONZE_EVENTS).count())   # 43200

# Idempotencia de Silver: re-ejecutar no cambia el conteo
antes = spark.read.parquet(SILVER_EVENTS).count()
run_silver_events()
despues = spark.read.parquet(SILVER_EVENTS).count()
print(f"Silver re-ejecutado -> antes={antes}  después={despues}")        # 42989 / 42989

Eventos de entrada: 43200
Pasaron el watermark: 727
Descartados por llegar tarde: 42473
% descartado: 98.32
Bronze con replay ordenado: 43200
Silver re-ejecutado -> antes=42989  después=42989
